<div style="text-align: justify; max-width: 90ch">

# Lecture 02b: A hand-built RAG, traced with MLflow

**Status: Mandatory reading.**

`lec_02a` turned texts into vectors and ranked them by meaning. Now put that to work: find the notes that answer a question and hand them to the model with the question.
That is **retrieval-augmented generation (RAG)**, the usual way to make a model answer from your own documents. We build it by hand,
with NumPy and `ollama`, so every step is visible, and trace it with MLflow, because a pipeline can fail in any step.

**What this notebook covers**

- Why a model cannot answer questions about your notes on its own.
- Loading the notes with `pathlib`, cut into sections.
- Embedding the sections once and caching the vectors.
- `retrieve` and `answer`: RAG in about twenty lines.
- **Decorators** and **MLflow tracing**: what every step received and returned.
- A wrong answer, diagnosed from its trace.

</div>

In [1]:
import os
import re
import time
from pathlib import Path

import mlflow
import numpy as np
import ollama
import pandas as pd
from dotenv import load_dotenv
from mlflow.entities import SpanType

from llm_course.checks import check_mlflow

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-02-rag"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. The model does not know your notes

Ask the chat model about the course itself. It has never seen these notes, and even a model that had could not tell you which lecture it read something in.
`ask` is the small wrapper from `lec_01c`, with a larger context window (`num_ctx`, `lec_01a`) because later prompts carry several pages of notes.

</div>

In [3]:
client = ollama.Client(host=OLLAMA_HOST)


def ask(prompt):
    """Send one prompt to the chat model and return the answer text."""
    response = client.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 200, "num_ctx": 8192},
    )
    return response.message.content.strip()

In [4]:
# WRONG: ask about the course without giving the model the notes
print(
    ask(
        "Which lecture of the Python course covers tracking experiments with MLflow? Answer in one sentence."
    )
)

The lecture that covers tracking experiments with MLflow is the one titled "Tracking Experiments with MLflow" in the Python course.


<div style="text-align: justify; max-width: 90ch">

Read the answer: does it name a lecture, and could it know? The notes teach MLflow tracking in lecture 13 (`lec_13c`); whatever the model names, it is a guess, because it does what it was trained to do:
continue the text plausibly. The fix is not a bigger model but **giving it the text**. RAG in three steps:

```text
question ──► Retrieve the most similar notes (embeddings, lec_02a)
         ──► Augment the prompt: instructions + notes + question
         ──► Generate the answer with the chat model
```

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Loading the notes

The notes are Markdown files in `corpus/uoa_py_course/`, one folder per lecture of the Python course. We use lectures 10 to 13 (k-NN, regression, classification, pipelines and MLflow):
about 40 files, realistic and quick to embed. `lec_02d` indexes all thirteen.

**`pathlib`** is the standard library's way to handle file paths. A `Path`:

- joins with `/`
- lists files with `glob`
- reads a file with `read_text()`

</div>

In [5]:
NOTES_DIR = REPO / "corpus" / "uoa_py_course"

files = []

for lecture in [10, 11, 12, 13]:
    files += sorted((NOTES_DIR / f"lecture_{lecture}").glob("*.md"))

print(len(files), "files, for example", files[0].relative_to(NOTES_DIR))
print(f"{sum(len(f.read_text(encoding='utf-8')) for f in files):,} characters")

41 files, for example lecture_10/goals_10.md
754,874 characters


<div style="text-align: justify; max-width: 90ch">

A whole file is too big to be "the relevant text": a lecture covers ten things, a question asks about one. We cut every file just before each `## ` heading, so each piece is one **section** of a lecture.
`lec_02c` names this properly, *chunking*, and does it better.

</div>

In [6]:
sections = []

for path in files:
    text = path.read_text(encoding="utf-8")
    for part in re.split(r"(?m)^(?=## )", text):  # cut before every line starting "## "
        if part.strip():
            sections.append(
                {
                    "source": str(path.relative_to(NOTES_DIR)),
                    "heading": next(
                        (line for line in part.splitlines() if line.startswith("#")), ""
                    )[:80],
                    "text": part,
                }
            )

print(len(sections), "sections")

pd.DataFrame(sections)[["source", "heading"]].sample(5, random_state=0)

414 sections


,source,heading
356,lecture_13/lec_13c_mlflow_tracking_basics.md,## Autologging: let MLflow log for you
170,lecture_11/lec_11e_polynomial_regression.md,## E. Picking `alpha` honestly — train / valid...
224,lecture_12/lec_12a_logistic_regression_intuiti...,## §E. A small real binary example — heart-dis...
331,lecture_13/lec_13a_gridsearchcv_hyperparameter...,## Recap and what's next
306,lecture_13/lec_13_exercises.md,## Section 1 - Hyperparameter search


<div style="text-align: justify; max-width: 90ch">

## 3. Embed once, keep the vectors

Every section goes through the embedding model once. That is the slow step of any RAG system, so the vectors are saved to `data/` and loaded on the next run (delete the file to recompute).
One limit appears here: the embedding model has a context window too, 2048 tokens, and a few sections are longer; Ollama would silently embed only their beginning (`lec_02c` shows it).
For now each section is cut to its first 4,000 characters; `lec_02c` fixes this properly. In example runs the first run took seconds on a GPU and a few minutes on a CPU.

</div>

In [7]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

MAX_CHARS = 4000  # stay inside the embedding model's 2048-token window (lec_02c)
vectors_path = DATA_DIR / "lec_02b_section_vectors.npy"
DATA_DIR.mkdir(exist_ok=True)

if vectors_path.exists() and len(np.load(vectors_path)) == len(sections):
    vectors = np.load(vectors_path)
else:
    started = time.time()
    embedded = []

    for start in range(0, len(sections), 32):
        batch = [s["text"][:MAX_CHARS] for s in sections[start : start + 32]]
        embedded += client.embed(model=EMBED_MODEL, input=batch).embeddings

    vectors = np.array(embedded)
    np.save(vectors_path, vectors)
    print(f"embedded {len(vectors)} sections in {time.time() - started:.0f} s")

vectors = vectors / np.linalg.norm(
    vectors, axis=1, keepdims=True
)  # length 1: dot = cosine

vectors.shape

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


(414, 768)

<div style="text-align: justify; max-width: 90ch">

## 4. Retrieve, then answer

`retrieve` is the ranking of `lec_02a` at the scale of the notes:

- embed the question
- one matrix product for all the cosines
- keep the best `k`

`answer` puts the retrieved sections into the prompt with their file names, so the model can say where its answer comes from.

</div>

In [8]:
def retrieve(question, k=3):
    """Return the k sections most similar to the question, best first."""
    question_vector = np.array(
        client.embed(model=EMBED_MODEL, input=question).embeddings[0]
    )
    scores = vectors @ (question_vector / np.linalg.norm(question_vector))
    best = np.argsort(-scores)[:k]
    return [{**sections[i], "score": round(float(scores[i]), 3)} for i in best]


pd.DataFrame(retrieve("Which lecture covers tracking experiments with MLflow?"))[
    ["score", "source", "heading"]
]

,score,source,heading
0,0.776,lecture_13/lec_13c_mlflow_tracking_basics.md,# Lecture 13c: Tracking Experiments with MLflow
1,0.733,lecture_13/lec_13d_mlflow_evaluation_registry.md,# Lecture 13d: Model Evaluation Gates and the ...
2,0.731,lecture_13/goals_13.md,"# Lecture 13: Automation & MLOps — Pipelines, ..."


In [9]:
PROMPT = """Answer the question using only the course notes below.
If the notes do not contain the answer, say so. Name the source file you used.

Notes:
{notes}

Question: {question}"""


def answer(question, k=3):
    """Retrieve k sections, put them into the prompt, and ask the chat model."""
    found = retrieve(question, k)
    notes = "\n\n".join(
        f"[source: {s['source']}]\n{s['text'][:MAX_CHARS]}" for s in found
    )
    return ask(PROMPT.format(notes=notes, question=question))

In [10]:
# RIGHT: the same question as section 1, with the notes in the prompt
print(answer("Which lecture covers tracking experiments with MLflow?"))

The lecture that covers tracking experiments with MLflow is **Lecture 13c**. The source file for this lecture is `lec_13c_mlflow_tracking_basics.md`.


<div style="text-align: justify; max-width: 90ch">

Same model, same question as section 1; the only change is that the prompt now carries the retrieved sections, listed in the table above.
Check the answer against them: does it name a file, and is that file one of the three? Now two harder questions. The first is one any data scientist would ask:

</div>

In [11]:
print(answer("What does GridSearchCV do?"))

The `GridSearchCV` is a method in scikit-learn that automatically searches through a predefined grid of hyperparameters for a given estimator. It uses cross-validation to evaluate each combination of hyperparameters and selects the combination that yields the best performance.

### Key Points:
- **Cross-validation**: `GridSearchCV` performs cross-validation for every combination in the grid, not just a single validation split. This helps in avoiding overfitting and provides a more reliable estimate of model performance.
- **Grid Search**: It searches through a grid of hyperparameters (defined by a `param_grid`), which is a dictionary of parameter names to lists of values.
- **Refit**: After fitting, `GridSearchCV` refits the best model on the entire training dataset with the best parameters, allowing for evaluation on new data.

### Source File:
`lec_13a_gridsearchcv_hyperparameter_tuning.md`

### Summary:
`GridSearchCV` is used to tune model hyper


<div style="text-align: justify; max-width: 90ch">

Depending on the run, the machine and the model versions, this answer explains `GridSearchCV` or says that the notes do not contain it.
Even a good answer proves little here: the model met `GridSearchCV` in its training data, so it can answer without the notes.
The second question has an answer only the notes know. `lec_12a` lowers the decision threshold of a heart-disease screening model from 0.5 to 0.3 and reports,
in a table, the recall it reaches:

</div>

In [12]:
HEART_QUESTION = "In the heart-disease screening example, what recall does the model reach at threshold 0.3?"

print(answer(HEART_QUESTION))

The question asks for the recall of the model at a threshold of 0.3 in the heart-disease screening example. 

From the provided notes, we can see that the model's performance is evaluated using the `recall` metric in the `GridSearchCV` with the `scoring="recall"` parameter. The notes also mention that the model's best parameters are such that `C = 1.0`, and the best mean CV recall is reported as `0.853` (rounded to three decimal places).

However, the question specifically asks for the recall at a threshold of 0.3. The notes do not explicitly state the recall at a specific threshold, but they do mention that the model's best parameters are such that `C = 1.0`, and the best mean CV recall is `0.853`.

Since the question is about the recall at a specific threshold (0.3), and the notes do not provide this specific value,


<div style="text-align: justify; max-width: 90ch">

The notes' answer is **0.79** (0.53 at the default 0.5). Read the answer above against it: does it state 0.79, give another number, or say that the notes do not contain it?

### ⏸ Pause point

Before reading on: which step failed?

- **The retriever** may have found the wrong sections.
- **The prompt** may have lost them.
- **The model** may have ignored them.

The final answer alone cannot tell you. This is the problem **tracing** solves.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Tracing each step with MLflow

In lecture 13 of the Python course MLflow recorded *runs* of a training job. For an application built of steps it records **traces** ([MLflow tracing](https://mlflow.org/docs/latest/genai/tracing/)):

| Term | Meaning |
| --- | --- |
| **trace** | one end-to-end call of your application, for example one `answer(...)` |
| **span** | one step inside it, with its inputs, outputs and duration; spans nest inside each other |
| **span type** | the kind of step: `RETRIEVER`, `CHAT_MODEL`, `CHAIN` (a pipeline of steps) |

Start the tracking server as in guide `02a_mlflow_server` (in this lecture's `infra_tools/`); the check below says so if it is not running.

</div>

In [13]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook

MLflow OK at http://127.0.0.1:5010


<div style="text-align: justify; max-width: 90ch">

A **decorator** is a line starting with `@` just above a `def`. It wraps the function, so every call goes through the wrapper first.
`@mlflow.trace` is such a wrapper: it records each call as a span, with the arguments as inputs and the return value as outputs.
Writing `@mlflow.trace` above `def f` is shorthand for `f = mlflow.trace(f)`, which lets us wrap the three functions we already have ([manual tracing](https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/manual-tracing/)):

</div>

In [14]:
# Run once after section 4: running this cell again would wrap the functions twice.
retrieve = mlflow.trace(retrieve, span_type=SpanType.RETRIEVER)
ask = mlflow.trace(ask, span_type=SpanType.CHAT_MODEL)
answer = mlflow.trace(answer, span_type=SpanType.CHAIN)

print(answer(HEART_QUESTION))

The question asks for the recall of the model at a threshold of 0.3 in the heart-disease screening example. 

From the provided notes, we can see that the model's performance is evaluated using the `recall` metric in the `GridSearchCV` with the `scoring="recall"` parameter. The notes also mention that the model's best parameters are such that `C = 1.0`, and the best mean CV recall is reported as `0.853` (rounded to three decimal places).

However, the question specifically asks for the recall at a threshold of 0.3. The notes do not explicitly state the recall at a specific threshold, but they do mention that the model's best parameters are such that `C = 1.0`, and the best mean CV recall is `0.853`.

Since the question is about the recall at a specific threshold (0.3), and the notes do not provide this specific value,


In [15]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)

pd.DataFrame(
    [
        {
            "span": s.name,
            "type": s.span_type,
            "parent": next(
                (p.name for p in trace.data.spans if p.span_id == s.parent_id), ""
            ),
            "seconds": round((s.end_time_ns - s.start_time_ns) / 1e9, 2),
        }
        for s in trace.data.spans
    ]
)

,span,type,parent,seconds
0,answer,CHAIN,,1.64
1,retrieve,RETRIEVER,answer,0.01
2,ask,CHAT_MODEL,answer,1.50


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

- **Open** <http://127.0.0.1:5010>, experiment `llm-course-02-rag`, **Traces** tab, and click the newest trace: `answer` contains `retrieve` and `ask`.
- **Click `retrieve`**: its **outputs** are the sections the model was given. Is any of them from `lec_12a`?
- **Then click `ask`**: its input is the prompt the model *actually* received. Search it for 0.79.

The same outputs, read from code:

</div>

In [16]:
retriever_span = next(s for s in trace.data.spans if s.span_type == SpanType.RETRIEVER)

pd.DataFrame(retriever_span.outputs)[["score", "source", "heading"]]

,score,source,heading
0,0.731,lecture_12/lec_12a_logistic_regression_intuiti...,## §E. A small real binary example — heart-dis...
1,0.714,lecture_13/lec_13a_gridsearchcv_hyperparameter...,## Choosing the scoring metric
2,0.709,lecture_12/lec_12c_logistic_regression_assumpt...,## §E. Class imbalance and the 0.5-threshold trap


In [17]:
ask_span = next(s for s in trace.data.spans if s.span_type == SpanType.CHAT_MODEL)

print("0.79 in the prompt:", "0.79" in ask_span.inputs["prompt"])

0.79 in the prompt: False


In [18]:
heart_section = next(s for s in sections if "0.79" in s["text"])

print(heart_section["source"], "|", heart_section["heading"])
print(
    "0.79 is",
    heart_section["text"].index("0.79"),
    "characters in; MAX_CHARS =",
    MAX_CHARS,
)

lecture_12/lec_12a_logistic_regression_intuition.md | ## §E. A small real binary example — heart-disease screening
0.79 is 10096 characters in; MAX_CHARS = 4000


<div style="text-align: justify; max-width: 90ch">

Whether the retriever ranked `lec_12a`'s section among the top three depends on the embedding model and its version. The prompt check does not:
the answer sits past the 4,000-character cut of section 3, so the model never receives it, on any run. **Even when the retriever finds the right section, the prompt step fails:
the section is cut before the answer.** The model can only say that the notes do not contain it, or guess. Without the trace we might have blamed the model, or rewritten the prompt for nothing.

This is the habit for the rest of the course: when an answer is wrong, **read the trace before changing the code**.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Why whole sections fail

Both questions point at the unit we retrieve, the section:

- **Long sections are cut.** Whatever comes after the first 4,000 characters is neither embedded nor sent to the model, so no question about it can be answered.
  The heart-disease recall fails this way on every run.
- **One vector per section is a blur.** A long section mixes several topics, and an almost empty heading gets a generic vector close to every short question.
  A class name such as `GridSearchCV` also carries little meaning for a general-purpose embedding model (`lec_02a`), so a question that names one may or may not reach the right section:
  it depends on the wording, the model and its version.

The course fixes both:

- `lec_02c` cuts the notes into chunks that fit the embedding window and drops pieces too short to answer anything.
- `lec_02d` adds a keyword condition for exact names.
- `lec_03d` lets a model rewrite the question before searching.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A chat model cannot answer questions about notes it has never seen; asked anyway, it invents.
- **RAG**: retrieve the most similar sections, add them to the prompt, generate; the model can then name its source.
- Embedding the documents is the slow step: do it once and cache the vectors.
- A **decorator** wraps a function; `@mlflow.trace` records each call as a **span**, and one end-to-end call as a **trace**.
- A good answer may come from the model's training, not from the notes: check what was retrieved, not only the answer.
- When an answer is wrong, read the trace first: for the heart-disease question the failing step was the prompt, which lost the answer when the section was cut.

The section splitter is now `llm_course.corpus.load_sections`, so later notebooks rebuild these sections in one line.

**Next: `lec_02c`**, why the pieces of text matter, and how to cut them.

</div>